# E-Commerce Product Return Prediction — Colab Notebook

Run cells from top to bottom. Upload `ecommerce_return_mlops.zip` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose ecommerce_return_mlops.zip


In [ ]:
import zipfile, os
zip_name = next(name for name in uploaded if name.endswith(".zip"))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")
os.chdir("/content/ecommerce_return_mlops")
print("Project folder:", os.getcwd())


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import pandas as pd
df = pd.read_csv("data/ecommerce_returns.csv")
print("Dataset shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("missing_values"))
display(df["Return_Status"].value_counts())


In [ ]:
!python -m src.train


In [ ]:
import json, os
metrics = json.load(open("artifacts/metrics.json"))
print(json.dumps(metrics, indent=2))
assert os.path.exists("artifacts/best_model.joblib")


In [ ]:
!pytest -q


In [ ]:
import subprocess, time
api_process = subprocess.Popen(
    ["uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(3)
print("API running:", api_process.poll() is None)


In [ ]:
import requests
print("Health:", requests.get("http://127.0.0.1:8000/health").json())
sample = {
    "Product_Category": "Electronics",
    "Product_Price_INR": 24999,
    "Discount_Percent": 15,
    "Expected_Delivery_Days": 4,
    "Actual_Delivery_Days": 6,
    "Customer_Rating_PreOrder": 3.5,
    "Previous_Returns": 2,
    "Previous_Orders": 8,
    "Payment_Method": "UPI",
    "Damaged_Package_Flag": "No",
    "Package_Size": "Medium",
    "Promotion_Active": "Yes"
}
r = requests.post("http://127.0.0.1:8000/predict", json=sample)
print("Prediction status:", r.status_code)
print("Prediction:", r.json())


In [ ]:
api_process.terminate()
print("Stopped API process.")
